### Teste Feature engineering

In [1]:
from sklearn.model_selection import train_test_split

from src.features.build_pipe_fe import build_pipe_fe
from src.features.new_features import TimeFeatureStrategy, AgeCategoryFeatureStrategy, HourCategoryFeatureStrategy, \
    NewFeature
import pandas as pd

In [2]:
df_transactions = pd.read_csv('../data/raw/transactions.csv', encoding='utf-8')
df_customers = pd.read_csv('../data/raw/customers.csv', encoding='utf-8')
df = pd.merge(
    df_transactions,
    df_customers,
    left_on='sender_id',
    right_on='customer_id',
    how='left'
)


In [3]:
target = 'fraud'

X = df.drop(columns=[target])
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [4]:
features = [
    TimeFeatureStrategy(),
    AgeCategoryFeatureStrategy(),
    HourCategoryFeatureStrategy()
]

In [5]:
cat_cols = ['hour_category', 'age_category', 'account_type', 'gender', 'device_type']
num_cols = ['amount', 'age']

columns_drop = [
    'transaction_id', 'timestamp', 'sender_id', 'receiver_id',
    'customer_id', 'cpf', 'pix_key', 'hour_date', 'minute_date'
]

In [6]:
pipe_fe = build_pipe_fe(num_cols=num_cols, cat_cols=cat_cols, features=features, columns_drop=columns_drop)

In [7]:
X_resampled, y_resampled = pipe_fe.fit_resample(X_train, y_train)

In [8]:
print(f'Antes: {X_train.shape}, {y_train.shape}')
print(f'Depois SMOTE: {X_resampled.shape}, {y_resampled.shape}')

Antes: (12000, 13), (12000,)
Depois SMOTE: (22216, 16), (22216,)


In [10]:
pd.DataFrame(X_resampled).to_pickle('../data/processed/X_resampled.pkl')
pd.Series(y_resampled).to_pickle('../data/processed/y_resampled.pkl')